In [1]:
# Make JAVA_HOME visible to notebook kernel before Spark starts
import os

# Set JAVA_HOME env var to root folder of Java 17 installation
os.environ["JAVA_HOME"] = "/Users/jerry/.sdkman/candidates/java/17.0.13-tem"
# PATH is a colon-separated search list
# We add {JAVA_HOME}/bin folder to path so that Spark can find "java" program
# We prepend this to PATH and keep original path as suffix
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:{os.environ["PATH"]}'

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("FlightData2015")
    .master("local[*]") # use as many threads as logical CPU cores on this local device
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "5")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/30 10:36:50 WARN Utils: Your hostname, Jerrys-MacBook-Pro-40.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.246 instead (on interface en0)
26/05/30 10:36:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 10:36:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Read from a file

fp = "data/flight-data/csv/2015-summary.csv"
flight_data_2015 = (
    spark.read
    .option("inferSchema", "true")
    .option("header", "true")
    .csv(fp)
)

In [4]:
# Return first n rows
flight_data_2015.show()
print(flight_data_2015.take(3))

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Romania|   15|
|       United States|            Croatia|    1|
|       United States|            Ireland|  344|
|               Egypt|      United States|   15|
|       United States|              India|   62|
|       United States|          Singapore|    1|
|       United States|            Grenada|   62|
|          Costa Rica|      United States|  588|
|             Senegal|      United States|   40|
|             Moldova|      United States|    1|
|       United States|       Sint Maarten|  325|
|       United States|   Marshall Islands|   39|
|              Guyana|      United States|   64|
|               Malta|      United States|    1|
|            Anguilla|      United States|   41|
|             Bolivia|      United States|   30|
|       United States|           Paraguay|    6|
|             Algeri

### Narrow and Wide Operations
Spark splits data up into partitions, so one dataset is not processed as one big object. Each partition is processed by one task. By default, one task is processed by one thread process.

Some operations do not need to move data. They can be easily parallelized in that each partition does the same thing and the results can be aggregated. For example, `df.filter(df['price'] > 100)` can be done independently across partitions. These are called **narrow** operations.

Other operations require related rows to be together. For example, if we perform `df.groupBy("city").count()`, but the city "Atlanta" appears in multiple different partitions, data needs to be moved around. This redistribution across partitions is called a **shuffle**. Operations requiring shuffles are called **wide** operations.

A **stage** is a group of tasks that can be run without needing a shuffle in the middle. When optimizing computations, Spark improves ordering and forms stages around shuffle boundaries.

In [5]:
# Narrow operation
us_flight_ct = (
    flight_data_2015.filter(
        (flight_data_2015["ORIGIN_COUNTRY_NAME"] == "United States") &
        (flight_data_2015["DEST_COUNTRY_NAME"] == "United States")
    )
    .select("count")
)
us_flight_ct.explain()
us_flight_ct.show()

== Physical Plan ==
*(1) Project [count#19]
+- *(1) Filter (((isnotnull(ORIGIN_COUNTRY_NAME#18) AND isnotnull(DEST_COUNTRY_NAME#17)) AND (ORIGIN_COUNTRY_NAME#18 = United States)) AND (DEST_COUNTRY_NAME#17 = United States))
   +- FileScan csv [DEST_COUNTRY_NAME#17,ORIGIN_COUNTRY_NAME#18,count#19] Batched: false, DataFilters: [isnotnull(ORIGIN_COUNTRY_NAME#18), isnotnull(DEST_COUNTRY_NAME#17), (ORIGIN_COUNTRY_NAME#18 = Un..., Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/jerry/coding/pyspark-practice/data/flight-data/csv/2015-su..., PartitionFilters: [], PushedFilters: [IsNotNull(ORIGIN_COUNTRY_NAME), IsNotNull(DEST_COUNTRY_NAME), EqualTo(ORIGIN_COUNTRY_NAME,United..., ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:int>


+------+
| count|
+------+
|370002|
+------+



In [6]:
# Wide operation
# Exchange indicates a shuffle between partitions
sorted_flights = flight_data_2015.sort(flight_data_2015["count"])
sorted_flights.explain()
sorted_flights.show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [count#19 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(count#19 ASC NULLS FIRST, 5), ENSURE_REQUIREMENTS, [plan_id=71]
      +- FileScan csv [DEST_COUNTRY_NAME#17,ORIGIN_COUNTRY_NAME#18,count#19] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/jerry/coding/pyspark-practice/data/flight-data/csv/2015-su..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:int>


+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Estonia|    1|
|              Kosovo|      United States|    1|
|              Zambia|      United States|    1|
|       United States|   Papua New Guinea|    1|
|               Malta|      United States|    1|
|       United States|          Gibraltar|    1|
| 

### Lazy Computations and Optimization
When we work with data, we often perform a series of transformations before an "action". In PySpark, the engine performs lazy evaluation. When we say `.groupBy()`, nothing actually happens until we require some result to be returned or data to be written somewhere, such as calling `df.take(3)`.

This gives the engine the ability to wait until it sees the whole picture to then optimize computations.

Lazy transformations typically return another PySpark object, like a DataFrame. Actions vary in what they return (they may write a result to disk, return a computed value, etc.) but all force some kind of change.

For example,
```python
df = spark.read.parquet("flights/") # read a folder of parquet files into one DataFrame
result = df.select("origin", "price").filter(df["price"] > 100) # resulting dataframe of origin/price cols fulfilling this filter condition
result.show() # this is the action
```

May get optimized into:
```
Read only needed columns: origin, price
Try to apply the filter while reading the file, or push the two closer
```

In [7]:
popular_us_origin_flights = (
    flight_data_2015.filter(flight_data_2015["ORIGIN_COUNTRY_NAME"] == "United States")
    .select("DEST_COUNTRY_NAME", "count")
    .filter(flight_data_2015["count"] > 200)
)
popular_us_origin_flights.explain()

popular_us_origin_flights.show()

== Physical Plan ==
*(1) Project [DEST_COUNTRY_NAME#17, count#19]
+- *(1) Filter (((isnotnull(ORIGIN_COUNTRY_NAME#18) AND isnotnull(count#19)) AND (ORIGIN_COUNTRY_NAME#18 = United States)) AND (count#19 > 200))
   +- FileScan csv [DEST_COUNTRY_NAME#17,ORIGIN_COUNTRY_NAME#18,count#19] Batched: false, DataFilters: [isnotnull(ORIGIN_COUNTRY_NAME#18), isnotnull(count#19), (ORIGIN_COUNTRY_NAME#18 = United States)..., Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/jerry/coding/pyspark-practice/data/flight-data/csv/2015-su..., PartitionFilters: [], PushedFilters: [IsNotNull(ORIGIN_COUNTRY_NAME), IsNotNull(count), EqualTo(ORIGIN_COUNTRY_NAME,United States), Gr..., ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:int>


+--------------------+------+
|   DEST_COUNTRY_NAME| count|
+--------------------+------+
|          Costa Rica|   588|
|Turks and Caicos ...|   230|
|               Italy|   382|
|            Honduras|   362|
|         The Bahamas|   9

### Streaming Data
A typical DataFrame's input source is static in that it doesn't continuously change. This allows for batch-style actions to be run after a computation graph is defined.

A streaming DataFrame is any DataFrame which has a source that is unbounded. For example, it could be a file directory that is receiving new files. We can still write normal DataFrame transformations, but Spark executes them incrementally.
- `spark.readStream` instead of `spark.read`
- No option to set `.option("inferSchema", "true")` since Spark cannot read all data in any instance; incoming data is not guaranteed to be in the same schema as the first file seen
- After defining the computational graph, we need `result_df.writeStream...start()` as the singular stream action, instead of any batch action

In [8]:
static_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("data/retail-data/by-day")
)

static_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [9]:
from pyspark.sql.functions import window, round

# Goal is to obtain the total spending of customers within a particular window

# Select SQL expressions as columns using list of strings
# Imagine this as SELECT <expr> FROM static_df
(
    static_df.selectExpr(
        "CustomerID",
        "(UnitPrice * Quantity) AS total_cost",
        "InvoiceDate"
    ).groupBy("CustomerID", window("InvoiceDate", "1 day"))
    .sum("total_cost")
    .withColumn("sum(total_cost)", round("sum(total_cost)", 2))
    .show(5)
)

+----------+--------------------+---------------+
|CustomerID|              window|sum(total_cost)|
+----------+--------------------+---------------+
|   14075.0|{2011-12-04 18:00...|         316.78|
|   18180.0|{2011-12-04 18:00...|         310.73|
|   15358.0|{2011-12-04 18:00...|         830.06|
|   15392.0|{2011-12-04 18:00...|         304.41|
|   15290.0|{2011-12-04 18:00...|         263.02|
+----------+--------------------+---------------+
only showing top 5 rows


In [10]:
# Set up a streaming DataFrame
# Main differences is that it cannot infer schema, and we use readStream instead of read
# Creates a link from the directory to DataFrame object (i.e., defines source)
streaming_df = (
    spark.readStream
    .format("csv")
    .schema(static_df.schema) # cannot use .option("inferSchema", "true") with streaming
    .option("maxFilesPerTrigger", 1) # omitted in prod; only to make streaming slower for demo
    .option("header", "true")
    .load("data/retail-data/by-day")
)

streaming_df.isStreaming # attribute flag; same DataFrame object

True

In [11]:
# Define computation graph to obtain spending of customers on hourly basis
# Unlike regular DataFrames, show() or batch actions don't work since they assume a finite result
customer_spending_per_hour = (
    streaming_df.selectExpr(
        "CustomerID",
        "(UnitPrice * Quantity) AS total_cost",
        "InvoiceDate"
    ).groupBy("CustomerID", window("InvoiceDate", "1 day"))
    .sum("total_cost")
    .withColumn("sum(total_cost)", round("sum(total_cost)", 2))
)

# .start() returns a StreamingQuery object, which let's us check status of running query
streaming_query = (
    customer_spending_per_hour.writeStream
    .format("memory") # creates temp SQL view
    .queryName("customer_purchases") # name of in-memory table
    .outputMode("complete") # whole result table is emitted (written to sink) per trigger
    .start() # streaming action
)

26/05/30 10:37:00 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/s0/zq5m1f7578s12jw_8klt90dm0000gn/T/temporary-4711754e-06fb-4840-96d7-1ed9fa6cc670. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/30 10:37:00 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [12]:
spark.stop()

26/05/30 10:37:00 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/05/30 10:37:00 ERROR MicroBatchExecution: Query customer_purchases [id = 37b3892e-1c29-43ef-97db-46e0d8792a9a, runId = 1ed48ad5-d019-4054-ae35-67132f2b57cf] terminated with error
org.apache.spark.SparkException: [INTERNAL_ERROR] The Spark SQL phase planning failed with an internal error. You hit a bug in Spark or the Spark plugins you use. Please, report this bug to the corresponding communities or vendors, and provide the full stack trace. SQLSTATE: XX000
	at org.apache.spark.SparkException$.internalError(SparkException.scala:107)
	at org.apache.spark.sql.execution.QueryExecution$.toInternalError(QueryExecution.scala:706)
	at org.apache.spark.sql.execution.QueryExecution$.withInternalError(QueryExecution.scala:719)
	at org.apache.spark.sql.execution.QueryExecution.$anonfun$executePhase$1(QueryExecution.scala:330)
	at org.apache.spark.sql.SparkSession.withActive(SparkSession.s